In [2]:
import os
import shutil
from datetime import datetime, timedelta
import datetime as dt
from openpyxl import load_workbook
from openpyxl.styles import Font
import pandas as pd

In [3]:
df1 = pd.read_csv('../../账户数据对接/消费对接(勿删).csv')
df1['大搜消费'] = df1['总消费2023Q4'] - df1['原生自主投放总消费2023Q4'] - df1['凤巢优惠券消费2023Q4'] - df1[
    '聚屏平台合约消费2023Q4'] - df1['聚屏平台合约消费2023Q4'] - df1['软植互选消费2023Q4'] - df1[
                      '度星选-软植互选-消费2023Q4']
df1['信息流消费'] = df1['原生自主投放总消费2023Q4'] - df1['原生CPC优惠券消费2023Q4'] - df1['原生CPM优惠券消费2023Q4']

In [4]:
# 上周日的日期
current_date = datetime.now()
one_weeks_ago = current_date - timedelta(days=current_date.weekday() + 1)
filename = './翼百信&布瑞泽数据跟进{}.xlsx'.format(one_weeks_ago.strftime('%m%d'))
ybx = pd.read_excel(filename, sheet_name='翼百信订单列表')
brz = pd.read_excel(filename, sheet_name='布瑞泽订单列表')
filename

'./翼百信&布瑞泽数据跟进1015.xlsx'

In [5]:
brz_original = brz.rename(
    columns={'开户账号': '账户名称', 'Q3大搜消费': '大搜消费', 'Q3信息流消费': '信息流消费', 'Q3总消费': '总消费'})
ybx_original = ybx.rename(
    columns={'开户账号': '账户名称', 'Q3大搜消费': '大搜消费', 'Q3信息流消费': '信息流消费', 'Q3总消费': '总消费'})
brz = brz_original[['账户名称', '账户状态', '大搜消费', '信息流消费']]
ybx = ybx_original[['账户名称', '账户状态', '大搜消费', '信息流消费']]
df1 = df1[['账户名称', '账户状态', '大搜消费', '信息流消费']]

In [6]:
brz = pd.merge(brz, df1, on='账户名称', how='left')
brz = brz.drop(['账户状态_x', '大搜消费_x', '信息流消费_x'], axis=1)
brz.columns = ['账户名称', '账户状态', '大搜消费', '信息流消费']
brz['总消费'] = brz['大搜消费'] + brz['信息流消费']
brz

,账户名称,账户状态,大搜消费,信息流消费,总消费
0,村秀花卉,该用户被拒绝,0.00,0.0,0.00
1,福建向大大服饰,该用户被拒绝,0.00,0.0,0.00
2,众易居,该用户被拒绝,0.00,0.0,0.00
3,厦门越懿网络,该用户被拒绝,0.00,0.0,0.00
4,厦门耘途教育,该用户被拒绝,0.00,0.0,0.00
...,...,...,...,...,...
652,数之能,用户正常生效,187.87,0.0,187.87
653,美弗仪,用户正常生效,862.56,0.0,862.56
654,君盛达,用户正常生效,1740.57,0.0,1740.57
655,招商加盟-小堡日记,用户正常生效,0.00,0.0,0.00


In [7]:
brz = brz.dropna(subset=['账户名称']).fillna("")
brz

,账户名称,账户状态,大搜消费,信息流消费,总消费
0,村秀花卉,该用户被拒绝,0,0,0
1,福建向大大服饰,该用户被拒绝,0,0,0
2,众易居,该用户被拒绝,0,0,0
3,厦门越懿网络,该用户被拒绝,0,0,0
4,厦门耘途教育,该用户被拒绝,0,0,0
...,...,...,...,...,...
652,数之能,用户正常生效,187.87,0,187.87
653,美弗仪,用户正常生效,862.56,0,862.56
654,君盛达,用户正常生效,1740.57,0,1740.57
655,招商加盟-小堡日记,用户正常生效,0,0,0


In [8]:
ybx = pd.merge(ybx, df1, on='账户名称', how='left')
ybx = ybx.drop(['账户状态_x', '大搜消费_x', '信息流消费_x'], axis=1)
ybx.columns = ['账户名称', '账户状态', '大搜消费', '信息流消费']
ybx['总消费'] = ybx['大搜消费'] + ybx['信息流消费']

In [9]:
ybx = ybx.dropna(subset=['账户名称']).fillna('')
ybx

,账户名称,账户状态,大搜消费,信息流消费,总消费
0,黄春伟,,,,
1,容利洁,该用户被拒绝,0,0,0
2,游互信息,该用户被拒绝,0,0,0
3,尊尚时计贸易,该用户被拒绝,0,0,0
4,国域股份,该用户被拒绝,0,0,0
...,...,...,...,...,...
326,邦唱网络科技,用户正常生效,136.92,0,136.92
327,闽粤涂料,用户正常生效,0,0,0
328,厦门优莱柏,用户正常生效,2086.21,0,2086.21
329,厦门贝科,用户正常生效,136.97,0,136.97


In [10]:
workbook = load_workbook(filename)
# 打开sheet名为'厦门总代理'的sheet
sheet_ybx = workbook['翼百信订单列表']
sheet_brz = workbook['布瑞泽订单列表']
sheet_tj = workbook['2023Q4任务完成情况']

In [47]:
for row_index, row in enumerate(sheet_ybx.iter_rows(min_row=2, max_row=sheet_ybx.max_row, values_only=True)):
    temp = (ybx[ybx['账户名称'] == str(row[3])]['账户状态'].empty & ybx[ybx['账户名称'] == str(row[3])][
        '大搜消费'].empty & ybx[ybx['账户名称'] == str(row[3])]['信息流消费'].empty &
            ybx[ybx['账户名称'] == str(row[3])]['总消费'].empty)
    if not temp:
        # row是第几行，col列
        sheet_ybx.cell(row=row_index + 2, column=9).value = ybx[ybx['账户名称'] == str(row[3])]['账户状态'].iloc[0]
        sheet_ybx.cell(row=row_index + 2, column=10).value = ybx[ybx['账户名称'] == str(row[3])]['大搜消费'].iloc[0]
        sheet_ybx.cell(row=row_index + 2, column=11).value = ybx[ybx['账户名称'] == str(row[3])]['信息流消费'].iloc[0]
        sheet_ybx.cell(row=row_index + 2, column=12).value = ybx[ybx['账户名称'] == str(row[3])]['总消费'].iloc[0]
workbook.save(filename)

In [48]:
for row_index, row in enumerate(sheet_brz.iter_rows(min_row=2, max_row=sheet_brz.max_row, values_only=True)):
    print(brz[brz['账户名称'] == str(row[3])])
    temp = (brz[brz['账户名称'] == str(row[3])]['账户状态'].empty & brz[brz['账户名称'] == str(row[3])][
        '大搜消费'].empty & brz[brz['账户名称'] == str(row[3])]['信息流消费'].empty &
            brz[brz['账户名称'] == str(row[3])]['总消费'].empty)
    if not temp:
        # row是第几行，col列
        sheet_brz.cell(row=row_index + 2, column=9).value = brz[brz['账户名称'] == str(row[3])]['账户状态'].iloc[0]
        sheet_brz.cell(row=row_index + 2, column=10).value = brz[brz['账户名称'] == str(row[3])]['大搜消费'].iloc[0]
        sheet_brz.cell(row=row_index + 2, column=11).value = brz[brz['账户名称'] == str(row[3])]['信息流消费'].iloc[0]
        sheet_brz.cell(row=row_index + 2, column=12).value = brz[brz['账户名称'] == str(row[3])]['总消费'].iloc[0]
workbook.save(filename)

     账户名称    账户状态 大搜消费 信息流消费 总消费
0    村秀花卉  该用户被拒绝    0     0   0
173  村秀花卉  该用户被拒绝    0     0   0
      账户名称    账户状态 大搜消费 信息流消费 总消费
1  福建向大大服饰  该用户被拒绝    0     0   0
  账户名称    账户状态 大搜消费 信息流消费 总消费
2  众易居  该用户被拒绝    0     0   0
     账户名称    账户状态 大搜消费 信息流消费 总消费
3  厦门越懿网络  该用户被拒绝    0     0   0
     账户名称    账户状态 大搜消费 信息流消费 总消费
4  厦门耘途教育  该用户被拒绝    0     0   0
   账户名称 账户状态 大搜消费 信息流消费 总消费
5  入款老户                    
    账户名称    账户状态 大搜消费 信息流消费 总消费
6  九龙星石业  该用户被拒绝    0     0   0
   账户名称    账户状态 大搜消费 信息流消费 总消费
7  晋江腾达  该用户被拒绝    0     0   0
     账户名称    账户状态 大搜消费 信息流消费 总消费
8  福建浔兴拉链  该用户被拒绝    0     0   0
     账户名称    账户状态 大搜消费 信息流消费 总消费
9  漳州祥英妇产  该用户被拒绝    0     0   0
    账户名称    账户状态 大搜消费 信息流消费 总消费
10  泉州蜜鹊  该用户被拒绝    0     0   0
     账户名称    账户状态 大搜消费 信息流消费 总消费
11  福建九龙星  该用户被拒绝    0     0   0
    账户名称 账户状态 大搜消费 信息流消费 总消费
12     -                    
16     -                    
17     -                    
18     -                    
20     -                    
21     -               

In [49]:
brz[brz["账户名称"] == 'xm群隆']

,账户名称,账户状态,大搜消费,信息流消费,总消费
656,xm群隆,用户正常生效,59,0,59


In [50]:
# 获取当前日期的月份
today = datetime.today()
current_month = today.month

# 计算当前日期所在季度的第一个月
quarter_start_month = ((current_month - 1) // 3) * 3 + 1
quarter_start_date = datetime(today.year, quarter_start_month, 1)
last_quarter_date = datetime(today.year, quarter_start_month - 3, 1)
last_quarter_date

datetime.datetime(2023, 7, 1, 0, 0)

In [51]:
brz_original['提单时间'] = pd.to_datetime(brz_original['提单时间'], format='%Y/%m/%d %H:%M:%S', errors='coerce')
ybx_original['提单时间'] = pd.to_datetime(ybx_original['提单时间'], format='%Y/%m/%d %H:%M:%S', errors='coerce')
brz_original

,提单产品,公司名称,网址,账户名称,开户日期,一线客服,部门,sf账号,账户状态,大搜消费,...,加V费,实际到款总额,近七日日均,提单时间,当前状态,所属地区,部门.1,销售人员,备注,近七天消费
0,大搜,漳浦县官浔镇村秀花卉专业合作社,www.cunxiuhh.com,村秀花卉,2017-03-16 00:00:00,朱晓婷,失效挽救部,易尔通客服四部,该用户被拒绝,0.00,...,NaN,NaN,NaN,2017-02-27,NaN,NaN,NaN,NaN,NaN,0.00
1,大搜,福建向大大服饰有限公司,https://xdada.taobao.com,福建向大大服饰,2018-05-23 00:00:00,朱晓婷,失效挽救部,易尔通客服四部,该用户被拒绝,0.00,...,NaN,NaN,0.0,2018-04-12,NaN,NaN,NaN,NaN,NaN,0.00
2,大搜,厦门众易居网络科技有限公司,http://www.zhongyiju360.com,众易居,2015-11-16 00:00:00,纪荣裕,失效挽救部,易尔通客服五部,该用户被拒绝,0.00,...,NaN,NaN,0.0,2018-05-16,NaN,NaN,NaN,NaN,NaN,9.09
3,大搜,厦门越懿网络科技有限公司,http://jingjiatong.cn/,厦门越懿网络,2018-06-26 10:59:23,无,失效挽救部,易尔通客服六部,该用户被拒绝,0.00,...,NaN,NaN,0.0,2018-06-25,NaN,NaN,NaN,NaN,NaN,0.00
4,大搜,厦门耘途教育咨询有限公司,http://www.xmyuntu.cn/,厦门耘途教育,2018-06-26 10:59:59,苏晨,失效挽救部,xm复活001,该用户被拒绝,0.00,...,NaN,NaN,0.0,2018-06-25,erp无此单，ICRM记录为冲单，销售许新德,NaN,NaN,NaN,NaN,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
553,大搜,厦门数之能科技有限公司,www.szndata.com,数之能,2023-08-23 00:00:00,刘能水,维护部门,xm维护27,用户正常生效,159.51,...,600.0,21200.0,NaN,2023-08-21,财务核对中,厦门,布瑞泽一部,朱斌1,NaN,NaN
554,大搜,厦门美弗仪自动化科技有限公司,www.meifuyizdh.com,美弗仪,2023-09-18 10:32:28,黎江洪,新开部门,xm新开05,用户正常生效,672.80,...,7600.0,8200.0,NaN,2023-09-13,完成,厦门,布瑞泽三部,黄道炜,NaN,NaN
555,大搜,厦门君盛达会务服务有限公司,qianhu.wejianzhan.com,君盛达,2023-09-20 00:00:00,阮总,新开部门,xm新开10,用户正常生效,1639.78,...,600.0,12400.0,NaN,2023-09-18,完成,厦门,布瑞泽一部,朱斌1,NaN,NaN
556,信息流,厦门小堡日记餐饮管理有限公司,qianhu.wejianzhan.com,招商加盟-小堡日记,2023-10-09 00:00:00,周女士,新开部门,xm新开12,用户正常生效,0.00,...,0.0,4800.0,NaN,2023-09-28,质控审核中,厦门,布瑞泽二部,王伟1,2023Q3未审核通过，计入2022Q4,NaN


In [52]:
count_greater_than_quarter_start = len(brz_original[brz_original['提单时间'] > quarter_start_date])

In [53]:
brz_temp = brz_original[brz_original['提单时间'] > quarter_start_date]
brz_temp

,提单产品,公司名称,网址,账户名称,开户日期,一线客服,部门,sf账号,账户状态,大搜消费,...,加V费,实际到款总额,近七日日均,提单时间,当前状态,所属地区,部门.1,销售人员,备注,近七天消费
557,大搜,厦门群隆仪器有限公司,www.bizhongji.cn,xm群隆,2023-10-12 00:00:00,许先生,维护部门,xm维护28,用户正常生效,26.85,...,600.0,11200.0,NaN,2023-10-09,财务核对中,厦门,布瑞泽二部,王伟1,NaN,NaN


In [54]:
ybx_temp = ybx_original[(ybx_original['提单时间'] > quarter_start_date)]
ybx_temp_1 = ybx_original[(ybx_original['提单时间'] > last_quarter_date) & (~ybx_original['备注'].isna())]
ybx_temp_2 = pd.concat([ybx_temp, ybx_temp_1])
ybx_temp_2

,提单产品,公司名称,网址,账户名称,开户日期,一线客服,部门,sf账号,账户状态,大搜消费,...,服务费,加V,实际到款总额,近7日日均,提单时间,当前状态,所属地区,部门.1,销售人员,备注
133,信息流,厦门推闻网络科技有限公司,qianhu.wejianzhan.com,NaN,NaT,黄总,NaN,NaN,NaN,0.0,...,2400.0,600.0,10900.0,NaN,2023-09-25,分配客服中,厦门,翼百信1,杨彩云,2023Q3未审核通过，计入2022Q4


In [55]:
# 本季度
brz_temp = brz_original[(brz_original['提单时间'] > quarter_start_date)]
# 上季度加入
brz_temp_1 = brz_original[(brz_original['提单时间'] > last_quarter_date) & (~brz_original['备注'].isna())]
# 合并
brz_temp_2 = pd.concat([brz_temp, brz_temp_1])
brz_temp_2

,提单产品,公司名称,网址,账户名称,开户日期,一线客服,部门,sf账号,账户状态,大搜消费,...,加V费,实际到款总额,近七日日均,提单时间,当前状态,所属地区,部门.1,销售人员,备注,近七天消费
557,大搜,厦门群隆仪器有限公司,www.bizhongji.cn,xm群隆,2023-10-12 00:00:00,许先生,维护部门,xm维护28,用户正常生效,26.85,...,600.0,11200.0,NaN,2023-10-09,财务核对中,厦门,布瑞泽二部,王伟1,NaN,NaN
556,信息流,厦门小堡日记餐饮管理有限公司,qianhu.wejianzhan.com,招商加盟-小堡日记,2023-10-09 00:00:00,周女士,新开部门,xm新开12,用户正常生效,0.00,...,0.0,4800.0,NaN,2023-09-28,质控审核中,厦门,布瑞泽二部,王伟1,2023Q3未审核通过，计入2022Q4,NaN


In [56]:
# Q4审核通过
brz_1 = brz_temp[(brz_temp['当前状态'] == '财务核对中') | (brz_temp['当前状态'] == '完成')]['当前状态'].count()
# Q3提单Q4审核通过
brz_2 = brz_temp_1[(brz_temp_1['当前状态'] == '财务核对中') | (brz_temp_1['当前状态'] == '完成')]['当前状态'].count()
# Q4提单总客户数
brz_3 = brz_1 + brz_2
# 大搜实到金额
brz_4 = brz_temp_2['大搜到款金额'].sum() - len(brz_temp_2[brz_temp_2['大搜到款金额'] > 0].index) * 600
# 信息流实到金额
brz_5 = brz_temp_2['信息流到款金额'].sum() - len(brz_temp_2[brz_temp_2['信息流到款金额'] > 0].index) * 600
# 总提单客户数
brz_6 = brz_original.shape[0]
# 开户上线客户数
filtered_brz_original = brz_original[(brz_original['账户名称'].notna()) & (brz_original['账户名称'] != '-')]
brz_7 = filtered_brz_original.shape[0]


In [57]:
from datetime import datetime


def filter_by_quarter_month_ranges(dataframe):
    # 获取当前日期
    current_date = datetime.today()

    # 计算当前季度的第一个月
    quarter_start_month = ((current_date.month - 1) // 3) * 3 + 1

    # 计算月份范围
    start_month_1 = quarter_start_month
    end_month_1 = quarter_start_month + 1
    end_month_2 = quarter_start_month + 2
    print(start_month_1, end_month_1, end_month_2)

    # 筛选大于等于当前季度的第一个月，小于下个月的数据
    part_1 = dataframe[
        (dataframe['提单时间'].dt.month >= start_month_1) & (dataframe['提单时间'].dt.month < end_month_1)]

    # 筛选大于等于下个月，小于下下个月的数据
    part_2 = dataframe[(dataframe['提单时间'].dt.month >= end_month_1) & (dataframe['提单时间'].dt.month < end_month_2)]

    # 筛选大于等于下下个月，小于下个季度的第一个月的数据
    part_3 = dataframe[dataframe['提单时间'].dt.month >= end_month_2]

    return part_1, part_2, part_3


# 调用函数并得到筛选后的数据
part_1, part_2, part_3 = filter_by_quarter_month_ranges(brz_temp_2)

10 11 12


In [58]:
if current_month == quarter_start_month:
    brz_temp_2, mark = part_1, 21
elif current_month == quarter_start_month + 1:
    brz_temp_2, mark = part_2, 22
else:
    brz_temp_2, mark = part_3, 23

In [59]:
grouped = brz_temp_2.groupby('提单产品')

# 计算每个分组的数量
product_counts = grouped.size()
product_counts

提单产品
大搜    1
dtype: int64

In [71]:
# 审核通过开户(大搜+信息流)
brz_8 = brz_temp_2[
    (~brz_temp_2['账户名称'].isna()) & ((brz_temp_2['当前状态'] == '财务核对中') | (brz_temp_2['当前状态'] == '完成'))]
brz_8

,提单产品,公司名称,网址,账户名称,开户日期,一线客服,部门,sf账号,账户状态,大搜消费,...,加V费,实际到款总额,近七日日均,提单时间,当前状态,所属地区,部门.1,销售人员,备注,近七天消费
557,大搜,厦门群隆仪器有限公司,www.bizhongji.cn,xm群隆,2023-10-12 00:00:00,许先生,维护部门,xm维护28,用户正常生效,26.85,...,600.0,11200.0,NaN,2023-10-09,财务核对中,厦门,布瑞泽二部,王伟1,NaN,NaN


In [61]:
brz_9 = brz_temp_2[brz_temp_2['大搜消费'] > 0]['大搜消费'].count()
brz_10 = brz_temp_2[brz_temp_2['信息流消费'] > 0]['信息流消费'].count()

In [73]:
sheet_tj['C6'] = brz_1
sheet_tj['D6'] = brz_2
sheet_tj['B6'] = brz_3
sheet_tj['L6'] = brz_4
sheet_tj['M6'] = brz_5
sheet_tj['B12'] = brz_6
sheet_tj['C12'] = brz_7
sheet_tj['C' + str(mark)] = product_counts['大搜'] if '大搜' in product_counts.index else 0
sheet_tj['D' + str(mark)] = product_counts['信息流'] if '信息流' in product_counts.index else 0
sheet_tj['E' + str(mark)] = product_counts['大搜+信息流'] if '大搜+信息流' in product_counts.index else 0
sheet_tj['H' + str(mark)] = brz_8.shape[0]
sheet_tj['J' + str(mark)] = brz_9
sheet_tj['K' + str(mark)] = brz_10
workbook.save(filename)

In [65]:
'大搜' in product_counts.index

True

In [7]:
from datetime import datetime, timedelta

current_date = datetime.now()
current_date.weekday()

3

In [10]:
current_date - timedelta(days=current_date.weekday() + 8)

datetime.datetime(2023, 10, 8, 8, 49, 53, 103631)

In [24]:
import os
import re

# 定义文件夹路径
folder_path = './订单列表/'

# 获取文件夹中的所有文件名
file_names = os.listdir(folder_path)
for file_name in file_names:
    # 替换文件名中的中文括号为英文括号
    new_file_name = file_name.replace('（', '(').replace('）', ')').replace(' ', '')

    # 构建新的文件路径
    old_file_path = os.path.join(folder_path, file_name)
    new_file_path = os.path.join(folder_path, new_file_name)

    # 重命名文件
    os.rename(old_file_path, new_file_path)

# 定义一个正则表达式来匹配文件名中的数字
pattern = r'(\d+)'

# 存储匹配到的数字
numbers = []

for file_name in file_names:
    # 使用正则表达式查找匹配的数字
    match = re.search(pattern, file_name)
    if match:
        numbers.append(int(match.group(1)))

# 找到最大的数字
if numbers:
    max_number = max(numbers)
    result = f'订单列表({max_number}).xls'
    print(result)
else:
    print("未找到匹配的文件")



订单列表(34).xls


In [67]:
temp = pd.read_excel(folder_path + "订单列表(32).xls")


# 遇到空行时，修正df的columns
def fix_dataframe_columns(df):
    for col in df.columns:
        if col.startswith('Unnamed:'):
            df.columns = df.iloc[0]
            df = df[1:]
            return fix_dataframe_columns(df)
    return df


# 调用函数来修复列名
temp = fix_dataframe_columns(temp)

temp = temp[(temp['所属大区'] == '布瑞泽') | (temp['所属大区'] == '翼百信')]
temp.reset_index(drop=True, inplace=True)
temp

,当前状态,质控人员,提单时间,所属地区,所属大区,部门,销售人员,是否公司建站,公司名称,开户账号,...,客户名字,客户电话,签单金额,服务费,加V服务费,信息流,实到金额,G宝盆,移动RUL,审核意见
0,完成,苏丽珍,2023/9/13 10:48:18,厦门,布瑞泽,布瑞泽三部,黄道炜,NaN,厦门美弗仪自动化科技有限公司,美弗仪,...,黎江洪,18720024742,8800,1200,7600,0,8800,NaN,www.meifuyizdh.com,NaN
1,完成,苏丽珍,2023/9/18 14:16:43,厦门,布瑞泽,布瑞泽一部,朱斌1,NaN,厦门君盛达会务服务有限公司,君盛达,...,阮总,18559265039,13000,2400,600,0,13000,NaN,qianhu.wejianzhan.com,"2023-09-18 17:08 苏丽珍执行质控驳回,驳回原因:未发邮件，大搜+认证金额错..."


In [66]:
# for row_index, row in temp.iterrows():
#         temp
if (temp.iloc[0]['签单金额']>0 & temp.iloc[0]['信息流消费']>0):
    print(1)

KeyError: '大搜消费'

In [35]:
last_data_row = None
# 遍历sheet的每一行,返回有数据的最后一行
for row_number, row in enumerate(
        sheet_brz.iter_rows(min_row=1, max_row=sheet_brz.max_row, min_col=1, max_col=1, values_only=True), start=1):
    if row[0] is not None:
        last_data_row = row_number + 1
        
print('write_Index：{}'.format(last_data_row))

write_Index：560


In [12]:
llast_date = current_date - timedelta(days=current_date.weekday() + 8)
if not os.path.exists('布瑞泽提单消费监控{}.xlsx'.format(one_weeks_ago.strftime('%m%d'))):
    shutil.copy2('布瑞泽提单消费监控{}.xlsx'.format(llast_date.strftime('%m%d')),
                 '布瑞泽提单消费监控{}.xlsx'.format(one_weeks_ago.strftime('%m%d')))
source_file = "./布瑞泽提单消费监控1015.xlsx"
source_wb = load_workbook(source_file)
source_hz = source_wb['数据汇总']
# 获取工作簿对象
sheet_to_delete = '提单消费明细'
# 检查工作表是否存在
if sheet_to_delete in source_wb.sheetnames:
    # 删除工作表
    del source_wb[sheet_to_delete]

data_to_write_ybx = [cell.value for cell in sheet_tj[6]]
print(data_to_write_ybx)

#
# 将数据插入或覆盖到工作表中
for column, value in enumerate(data_to_write_ybx, 1):
    source_hz.cell(row=4, column=column, value=value)

source_hz['E4'] = source_hz['B4'].value + source_hz['D4'].value
source_hz['G4'] = brz['大搜消费'].replace('', 0).sum()
source_hz['H4'] = brz['信息流消费'].replace('', 0).sum()
source_hz['F4'] = brz['信息流消费'].replace('', 0).sum() + brz['大搜消费'].replace('', 0).sum()
source_hz['K4'] = source_hz['L4'].value + source_hz['M4'].value
# 
if '提单消费明细' not in source_wb.sheetnames:
    source_wb.create_sheet('提单消费明细')


# 保存更改后的工作簿
output_file = '布瑞泽提单消费监控{}.xlsx'.format(one_weeks_ago.strftime('%m%d'))
source_wb.save(output_file)

['布瑞泽', 1, 1, 1, '=B6+D6', '=G6+H6', '=G12', '=I12', '-', '-', '=L6+M6', 11200, 4800, '-', '-', None]
